# Training Model SCENTINEL
Melatih model AI untuk deteksi kelayakan makanan berbasis data sensor gas dan lingkungan (MQ-3, MQ-4, MQ-135, TGS2602, Suhu, Kelembapan).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
# Pastikan Anda sudah menjalankan serial_logger.py dan menghasilkan raw_data.csv
try:
    df = pd.read_csv('../datasets/raw_data.csv')
    print(f"Data berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom.")
    display(df.head())
except FileNotFoundError:
    print("File raw_data.csv tidak ditemukan. Harap kumpulkan data terlebih dahulu.")

In [ ]:
# 2. Preprocessing
# Hapus baris dengan nilai NaN (jika sensor sempat gagal baca)
df = df.dropna()

# Pisahkan fitur (X) dan label (y)
# Fitur: nilai raw dari MQ3, MQ4, MQ135, TGS2602 serta Suhu & Kelembapan
X = df[['mq3_raw', 'mq4_raw', 'mq135_raw', 'tgs2602_raw', 'temperature_c', 'humidity_pct']]

# Label: LAYAK -> 0, TIDAK_LAYAK -> 1
y = df['label'].apply(lambda x: 1 if x == 'TIDAK_LAYAK' else 0)

# Split dataset (80% data latih, 20% data uji)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standarisasi fitur agar memiliki skala yang sama (penting untuk banyak algoritma ML)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Distribusi kelas di Data Latih:", np.bincount(y_train))
print("Distribusi kelas di Data Uji:", np.bincount(y_test))

In [ ]:
# 3. Model Training (Menggunakan Random Forest)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data uji
y_pred = model.predict(X_test_scaled)

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['LAYAK', 'TIDAK_LAYAK'], yticklabels=['LAYAK', 'TIDAK_LAYAK'])
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['LAYAK', 'TIDAK_LAYAK']))

In [ ]:
# 4. Feature Importance
# Mengetahui sensor mana yang paling berpengaruh terhadap keputusan
feature_importances = model.feature_importances_
features = X.columns

plt.figure(figsize=(8, 5))
sns.barplot(x=feature_importances, y=features)
plt.title('Tingkat Kepentingan Sensor (Feature Importance)')
plt.show()

In [ ]:
# 5. Simpan Model & Scaler
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(model, '../models/scentinel_model.joblib')
joblib.dump(scaler, '../models/scentinel_scaler.joblib')
print("✅ Model dan Scaler berhasil disimpan di folder 'ml/models'")